# Vision Transformer (ViT)

自然言語で使われるTransformerは、画像処理にも応用されている。
## インプットレイヤー
画像を小さなパッチに分割し（ただマス目状に切り分けたもの）、Flattenで横に切断して横に並べる。線形、埋め込み。クラス埋め込み（画像情報を詰め込んだベクトルのこと）位置の情報も付与。どの1に注目すべきかの情報。
## Transformer encoder
Transformer encoderは、正規化、Multi-Head Attention、Feed Forward Networkを通して、画像の特徴を抽出する。MLPヘッドで分類などのタスクに利用する。
これらのパッチをトークンとして扱うことで、CNNに代わってTransformerの自己注意機構（Self-Attention）を利用して画像の特徴を捉えることができる。どの部分に注目すべきか、表現できる。これにより、従来のCNNベースのモデルよりも柔軟で強力な画像認識が可能となる。

In [ ]:
# Colab環境で必要なライブラリをインストール
!pip install vit_keras
!pip install tensorflow-addons

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.8/611.8 kB 9.9 MB/s eta 0:00:00


In [ ]:
# 学習に必要なライブラリを読み込み
import keras
from tensorflow.keras import datasets, layers, models
from vit_keras import vit
import tensorflow_addons as tfa

/usr/local/lib/python3.10/dist-packages/tensorflow_addons/utils/tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(


In [ ]:
# CIFAR-10データセットを読み込み（学習用とテスト用）
(x_train, y_train), (x_test, y_test) = datasets.cifar10.load_data()

170498071/170498071 [==============================] - 5s 0us/step


In [ ]:
# CIFAR-10は10クラス分類なのでクラス数を定義
num_classes = 10

# ラベルをワンホット表現へ変換（例: 3 -> [0,0,0,1,0,0,0,0,0,0]）
y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

In [ ]:
# 画像データを学習で扱いやすいfloat32型に変換
x_train = x_train.astype("float32")
x_test = x_test.astype("float32")

In [ ]:
# 255で割って、画素値を0-255から0-1へ正規化（学習を安定させるため）
x_train = x_train / 255.0
x_test = x_test / 255.0

In [ ]:
# ViT-B16モデルを作成
# image_size=32: CIFAR-10画像サイズに合わせる
# include_top=False: 既存の分類ヘッドは使わない（自分で付ける）
# pretrained_top=False: 事前学習ヘッドは利用しない
vit_model = vit.vit_b16(
    image_size=32,
    include_top=False,
    pretrained_top=False,
)

347502902/347502902 [==============================] - 2s 0us/step


/usr/local/lib/python3.10/dist-packages/vit_keras/utils.py:81: UserWarning: Resizing position embeddings from 24, 24 to 2, 2
  warnings.warn(


In [ ]:
# ViT本体の上に、10クラス分類用のDense層を追加
model = models.Sequential([
    vit_model,
    layers.Dense(num_classes, activation="softmax")
])

In [ ]:
# モデルの学習設定
# optimizer: 重み更新手法
# loss: 多クラス分類用の損失関数
# metrics: 学習中に表示する評価指標
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

In [ ]:
# モデルの層構造とパラメータ数を表示
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 vit-b16 (Functional)        (None, 768)               85651200  
                                                                 
 dense (Dense)               (None, 10)                7690      
                                                                 
Total params: 85658890 (326.76 MB)
Trainable params: 85658890 (326.76 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [ ]:
# モデルを学習
# batch_size=32: 32枚ずつ学習
# epochs=3: データ全体を3周
# validation_split=0.2: 学習データの20%を検証用に使用
history = model.fit(x_train, y_train, batch_size=32, epochs=3, validation_split=0.2)

Epoch 1/3
1250/1250 [==============================] - 125s 69ms/step - loss: 1.8052 - accuracy: 0.3299 - val_loss: 1.4955 - val_accuracy: 0.4627
Epoch 2/3
1250/1250 [==============================] - 84s 67ms/step - loss: 1.3944 - accuracy: 0.4988 - val_loss: 1.3373 - val_accuracy: 0.5169
Epoch 3/3
1250/1250 [==============================] - 84s 67ms/step - loss: 1.2586 - accuracy: 0.5501 - val_loss: 1.2489 - val_accuracy: 0.5592


In [ ]:
# テストデータで最終的な性能を評価
_, acc = model.evaluate(x_test, y_test)

313/313 [==============================] - 11s 36ms/step - loss: 1.2344 - accuracy: 0.5509


In [ ]:
# テスト精度を確認
acc

0.5508999824523926

In [ ]:
# テストデータに対するクラス確率を予測
y_pred = model.predict(x_test)

313/313 [==============================] - 13s 32ms/step


In [ ]:
# 予測結果の形を確認（件数, クラス数）
y_pred.shape

(10000, 10)

In [ ]:
# 1枚目画像の各クラスである確率を確認
y_pred[0, :]

array([0.04941141, 0.02359232, 0.13613138, 0.35197413, 0.14462954,
       0.16179183, 0.0391544 , 0.03269272, 0.04691428, 0.01370796],
      dtype=float32)

In [ ]:
# メモ: 追加実験用セル（学習率変更、エポック増加、データ拡張など）